# Домашнее задание №4

Материалом для практики послужит текст из [книги](https://fb2.top/pisyma-nikolay-erdman-angelina-stepanova-1928-1935-gg-457615), содержащей переписку писателя-драматурга Николая Эрдмана и его возлюбленной Ангелиной Степановой, актрисы МХАТа. 
В тексте есть сами письма (с 1928 по 1935 гг.), а также фрагменты из воспоминаний Степановой (опубликованных в 1995 году и относящихся к этому периоду).


Ваша задача — подготовить датасет писем для дальнейшего количественного анализа.

Необходимо сделать pandas датафрейм, в котором буду храниться следующие сведения (столбцы): 
- дата написания письма (желательно в формате YYYY-MM-DD), 
- автор (Степанова или Эрдман), 
- текст письма, 
- лемматизированный текст письма (без знаков препинания), 
- количество слов в письме. 

> Воспоминания сохранять в датафрейм НЕ НУЖНО!
> Сноски вида [23] и переносы строк нужно удалить из текста.

Чтобы извлечь письма и метаданные о них, нужно воспользоваться [регулярными выражениями](https://docs.python.org/3/library/re.html). 

Обратите внимание, что даты оформлены не единообразно. Продумайте все возможные варианты, перепроверьте данные. Адресант письма так же может быть указан не единообразно или не указан вовсе (но, например, есть адресат). Задача — извлечь метаданные для наибольшего количества писем.

После окончания работы над датасетом необходимо вывести df.info(), чтобы можно было посмотреть количество извлечённых дат, а также valuecounts() для столбца с адресантами.

### КРИТЕРИИ

| Что сделано | Баллы |
|---|---:|
| Извлечены тексты писем, собраны в датафрейм с указанными столбцами | 2 |
| Текст очищен и лемматизирован | 1 |
| Автор каждого письма определён: «Степанова» или «Эрдман» (включая случаи, когда автор указан не единообразно или восстанавливается через адресата) | 2 |
| Все имеющиеся даты приведены к единому формату (в оригинале даты не единообразны, например, «1928», «января 1928», «12.02.1928», «12 февраля 1928») | 3 |
| Датасет опубликован на GitHub в отдельном репозитории с файлом README.md (описание источника, структуры столбцов, методов извлечения и очистки, описаны принципы работы с отсутствующими значениями дат). Достаточно прикрепить ссылку на репозиторий. Датасет нужно опубликовать в формате .csv. | 2 |

Для поднятия настроения — басня Эрдмана:

```
СМЕТАНА.

Мы любим подмечать у недругов изъяны,
И направлять на них насмешки остриё.

Однажды молоко спросило у сметаны:
- Скажите, вы еда или питье?
Сметана молвила:
- Оставьте ваши шутки!
Действительно, я где-то в промежутке,
Но ведь важна не эта сторона
Всего важнее то, что я вкусна
И все, как правило, бывают мною сыты.

Вот так же точно и гермафродиты:
Тот, кто на свет их произвел,
Конечно, допустил ужасную небрежность.
Но ведь в конце-концов
существенен не пол,
А классовая принадлежность.
```

Читать другие басни: https://stihi.ru/diary/valery31/2014-10-11

In [1]:
import re
import pandas as pd

In [2]:
with open(r'C:\Users\krist\Desktop\Учеба_Программирование\erdman.txt', 'r', encoding='utf-8') as f:
    text = f.read()

blocks = text.split('\n* * *\n')

In [3]:
def strip_memoirs(block):
    marker = '[Воспоминания А. И. Степановой]'
    if marker in block:
        s = block.lstrip()
        if s.startswith(marker):
            return ''
        idx = block.find(marker)
        block = block[:idx]
    block = re.sub(r'\n\s*Они встретились в квартире Бориса Робертовича.*$', '', block, flags=re.S)
    return block.strip()


letters = []
for b in blocks:
    b2 = strip_memoirs(b)
    if b2:
        letters.append(b2)

In [ ]:
# извлечение автора!!!!!!!!!!!!!!!!!!!!!!!
def detect_author(block):
    lines = [ln.strip() for ln in block.splitlines() if ln.strip()]
    head = '\n'.join(lines[:12])
    tail_lines = lines[-6:]

    # подпись в конце
    sig = None
    for ln in reversed(tail_lines):
        ln_clean = re.sub(r'[\s\.\-–—]+$', '', ln).strip()
        if ln_clean in ['Лина', 'Л', 'Линуша']:
            sig = 'Степанова'
            break
        if ln_clean in ['Николай', 'Н', 'Коля']:
            sig = 'Эрдман'
            break
        # явные подписи в строке
        if re.search(r'(^|\s)Лина\.?(\s|$)', ln) and len(ln) < 80:
            if re.search(r'\bЛина\.?\s*$', ln) or ln.strip() in ['Лина', 'Лина.']:
                sig = 'Степанова'
                break
        if re.search(r'\bНиколай\b', ln):
            # подпись Николая (часто вместе с датой/приветами)
            if re.search(r'\bНиколай\.?\s*(\d|$|Привет|П\.|Целую)', ln) or len(ln) < 80:
                sig = 'Эрдман'
                break
        if re.search(r'^Н\.$', ln.strip()):
            sig = 'Эрдман'
            break
        if re.search(r'Целую\s+Коля', ln, flags=re.IGNORECASE) or re.search(r'\bЦК\b', ln):
            sig = 'Эрдман'
            break

    # адресат в начале
    to_stepanova = bool(re.search(r'А\.\s*О\.\s*Степановой|Ангелине\s+Осиповне\s+Степановой|для\s+А\.\s*О\.\s*Степановой', head))
    to_erdman = bool(re.search(r'Н\.\s*Р\.\s*Эрдману|Николаю\s+Робертовичу\s+Эрдману', head))
    if re.search(r'Томск\.\s*(До\s+востребования|Востребование)\.\s*(Н\.\s*Р\.\s*)?Эрдман', head, flags=re.IGNORECASE):
        to_erdman = True

    if sig:
        return sig
    if to_erdman and not to_stepanova:
        return 'Степанова'
    if to_stepanova and not to_erdman:
        return 'Эрдман'

    end = block.strip()[-200:]
    if re.search(r'\bЛина\.?\s*$', end):
        return 'Степанова'
    if re.search(r'\bНиколай\b', end.splitlines()[-1] if end.splitlines() else end):
        return 'Эрдман'
    if re.search(r'\bНиколай\.?\s*$', end) or re.search(r'(^|\s)Н\.\s*$', end):
        return 'Эрдман'
    if re.search(r'Целую\s+Коля', end, flags=re.IGNORECASE):
        return 'Эрдман'

    # спец.случаи
    if re.search(r'\bот ваш\. кор\.', block, flags=re.IGNORECASE) and re.search(r'Енисейск', block):
        return 'Эрдман'
    if re.search(r'после\s+«Булычева»', block) and re.search(r'пойду', block):
        return 'Степанова'

    # если в шапке явно адресовано Степановой
    if to_stepanova:
        return 'Эрдман'
    if to_erdman:
        return 'Степанова'
    return None

In [7]:
# извлечение и нормализация даты!!!!!!!!!!!!!!!!!
months = {
    'января': 1, 'февраля': 2, 'марта': 3, 'апреля': 4, 'мая': 5, 'июня': 6,
    'июля': 7, 'августа': 8, 'сентября': 9, 'октября': 10, 'ноября': 11, 'декабря': 12,
    'янв': 1, 'фев': 2, 'февр': 2, 'мар': 3, 'апр': 4, 'май': 5, 'мая': 5, 'июн': 6,
    'июл': 7, 'авг': 8, 'сен': 9, 'сент': 9, 'окт': 10, 'ноя': 11, 'нояб': 11, 'дек': 12,
}
romans = {'I':1,'II':2,'III':3,'IV':4,'V':5,'VI':6,'VII':7,'VIII':8,'IX':9,'X':10,'XI':11,'XII':12}
month_names_nom = {
    'январь':1,'февраль':2,'март':3,'апрель':4,'май':5,'июнь':6,
    'июль':7,'август':8,'сентябрь':9,'октябрь':10,'ноябрь':11,'декабрь':12
}

In [8]:
def norm_year(y):
    y = int(y)
    if y < 100:
        return 1900 + y
    return y


def make_date(y, m, d):
    if y is None or m is None:
        return None
    if d is None:
        d = 1
    try:
        return f"{int(y):04d}-{int(m):02d}-{int(d):02d}"
    except Exception:
        return None

In [9]:
def extract_date_info(block):
    # ищем в первых строках и по всему тексту
    lines = [ln.strip() for ln in block.splitlines() if ln.strip()]
    candidates = []
    text_all = '\n'.join(lines)

    # 1) [1 января 34 г.] / (25 апреля 34 г.) / 1 января 34 г. / 3 ноября 1933 г.
    for m in re.finditer(r'[\[\(]?\b(\d{1,2})\s+(января|февраля|марта|апреля|мая|июня|июля|августа|сентября|октября|ноября|декабря|янв\.?|февр?\.?|мар\.?|апр\.?|июн\.?|июл\.?|авг\.?|сент\.?|окт\.?|нояб\.?|дек\.?)\s+(\d{2,4})\s*(?:г\.?|года)?\b[\]\)]?', text_all, flags=re.IGNORECASE):
        d = int(m.group(1)); mon = months.get(m.group(2).lower().strip('.'), None); y = norm_year(m.group(3))
        if mon:
            candidates.append((y, mon, d, m.start(), 3))

    # 2) 10/V-34 г.  23/VI-34 г. 18/III-34 г.
    for m in re.finditer(r'\b(\d{1,2})/([IVX]+)[-–](\d{2,4})\s*(?:г\.?)?\b', text_all):
        d = int(m.group(1)); mon = romans.get(m.group(2).upper()); y = norm_year(m.group(3))
        if mon:
            candidates.append((y, mon, d, m.start(), 3))

    # 3) 20.01.34 г. 7.03.35 14.12.34 г.
    for m in re.finditer(r'\b(\d{1,2})[./](\d{1,2})[./](\d{2,4})\s*(?:г\.?)?\b', text_all):
        d = int(m.group(1)); mon = int(m.group(2)); y = norm_year(m.group(3))
        if 1 <= mon <= 12:
            candidates.append((y, mon, d, m.start(), 3))

    # 4) день + месяц без года: "5 ноября", "22 февраля"
    for m in re.finditer(r'(?m)^(\d{1,2})\s+(января|февраля|марта|апреля|мая|июня|июля|августа|сентября|октября|ноября|декабря)\s*$', text_all, flags=re.IGNORECASE):
        d = int(m.group(1)); mon = months.get(m.group(2).lower().strip('.'))
        if mon:
            candidates.append((None, mon, d, m.start(), 2))
    # в строке (не только отдельная строка), но без года рядом
    if not any(c[4] == 2 for c in candidates):
        for m in re.finditer(r'\b(\d{1,2})\s+(января|февраля|марта|апреля|мая|июня|июля|августа|сентября|октября|ноября|декабря)\b(?!\s+\d{2,4})', text_all, flags=re.IGNORECASE):
            d = int(m.group(1)); mon = months.get(m.group(2).lower().strip('.'))
            if mon:
                candidates.append((None, mon, d, m.start(), 1)) # избегание ложных срабатываний в середине текста? но у нас даты часто отдельно

    # 5) только месяц: "Ноябрь"
    if not candidates:
        for m in re.finditer(r'(?m)^(Январь|Февраль|Март|Апрель|Май|Июнь|Июль|Август|Сентябрь|Октябрь|Ноябрь|Декабрь)\.?\s*$', text_all, flags=re.IGNORECASE):
            mon = month_names_nom.get(m.group(1).lower())
            if mon:
                candidates.append((None, mon, None, m.start(), 1))

    if not candidates:
        return None

    # выбираем лучший, т.е. приоритет по полноте, затем самый ранний по позиции (часто дата вверху)
    candidates.sort(key=lambda x: (-x[4], x[3]))
    y, mon, d, pos, score = candidates[0]
    return {'y': y, 'm': mon, 'd': d}

In [10]:
# очистка текста письма!!!!!!!!!!!!!!!!
def clean_text(block):
    # убираем сноски [23]
    t = re.sub(r'\[\d+\]', '', block)
    # переносы строк -> пробелы
    t = re.sub(r'\s+', ' ', t).strip()
    return t

In [ ]:
# лемматизация!!!!!!!!!!
try:
    from razdel import tokenize
    import pymorphy3
    morph = pymorphy3.MorphAnalyzer()

    def lemmatize_text(text):
        words = [t.text for t in tokenize(text)]
        lemmas = []
        for w in words:
            w_clean = re.sub(r'[^A-Za-zА-Яа-яЁё0-9-]', '', w)
            if not w_clean:
                continue
            if not re.search(r'[A-Za-zА-Яа-яЁё]', w_clean):
                continue
            p = morph.parse(w_clean)[0]
            lemmas.append(p.normal_form)
        return ' '.join(lemmas)
except Exception:
    def lemmatize_text(text): # запасной вариант
        words = re.findall(r'[А-Яа-яЁёA-Za-z]+', text.lower())
        return ' '.join(words)

In [13]:
def count_words(text):
    words = re.findall(r'[А-Яа-яЁёA-Za-z0-9-]+', text)
    return len(words)

In [14]:
# !!!!!!!!!!сбор данных
rows = []
date_infos = []
for block in letters:
    author = detect_author(block)
    dinfo = extract_date_info(block)
    date_infos.append(dinfo)
    cleaned = clean_text(block)
    lemmas = lemmatize_text(cleaned)
    wc = count_words(cleaned)
    rows.append({
        'текст письма': cleaned,
        'автор': author,
        'лемматизированный текст письма': lemmas,
        'количество слов в письме': wc,
        '_dinfo': dinfo,
    })

In [ ]:
# восстановление года для дат без года
# сначала известные годы
known_years = [i for i, di in enumerate(date_infos) if di and di.get('y')]

def infer_year(idx):
    di = date_infos[idx]
    if di is None or di.get('y'):
        return di['y'] if di else None
    left = None    # ищем ближайшую известную дату слева 
    right = None    # ищем ближайшую известную дату справа
    for j in range(idx - 1, -1, -1):
        if date_infos[j] and date_infos[j].get('y'):
            left = (j, date_infos[j])
            break
    for j in range(idx + 1, len(date_infos)):
        if date_infos[j] and date_infos[j].get('y'):
            right = (j, date_infos[j])
            break
    # если обе известны
    if left and right:
        ly = left[1]['y']; ry = right[1]['y']
        if ly == ry: # если год одинаковый - берем его
            return ly
        m = di.get('m') # если разные, пробуем по месяцу относительно границ
        if idx - left[0] <= right[0] - idx:  # эвристика выбираем год ближайшего соседа
            return ly
        else:
            return ry
    if left:
        return left[1]['y']
    if right:
        return right[1]['y']
    return None

In [ ]:
# на этом этапе я чуток устала

In [ ]:
for _ in range(3):
    for i, di in enumerate(date_infos):
        if di and di.get('y') is None:
            y = infer_year(i)
            if y:
                di['y'] = y

final_rows = []
for i, row in enumerate(rows):
    di = row.pop('_dinfo')
    date_str = None
    if di:
        date_str = make_date(di.get('y'), di.get('m'), di.get('d'))
    final_rows.append({
        'дата написания письма': date_str,
        'автор': row['автор'],
        'текст письма': row['текст письма'],
        'лемматизированный текст письма': row['лемматизированный текст письма'],
        'количество слов в письме': row['количество слов в письме'],
    })

In [17]:
df = pd.DataFrame(final_rows, columns=[
    'дата написания письма',
    'автор',
    'текст письма',
    'лемматизированный текст письма',
    'количество слов в письме',
])

In [18]:
out_csv = r'C:\Users\krist\Desktop\Учеба_Программирование\erdman_letters.csv'
df.to_csv(out_csv, index=False, encoding='utf-8-sig')

print('Всего писем:', len(df))
print()
df.info()
print()
print('Количество писем по авторам:')
print(df['автор'].value_counts(dropna=False))
print()
print('Пропуски в датах:', df['дата написания письма'].isna().sum())
# показать первые строки для проверки
print('\nПримеры:')
print(df[['дата написания письма','автор','количество слов в письме']].head(12).to_string(index=False))
print('...')
print(df[['дата написания письма','автор','количество слов в письме']].tail(8).to_string(index=False))


Всего писем: 74

<class 'pandas.DataFrame'>
RangeIndex: 74 entries, 0 to 73
Data columns (total 5 columns):
 #   Column                          Non-Null Count  Dtype
---  ------                          --------------  -----
 0   дата написания письма           60 non-null     str  
 1   автор                           74 non-null     str  
 2   текст письма                    74 non-null     str  
 3   лемматизированный текст письма  74 non-null     str  
 4   количество слов в письме        74 non-null     int64
dtypes: int64(1), str(4)
memory usage: 271.0 KB

Количество писем по авторам:
автор
Эрдман       43
Степанова    31
Name: count, dtype: int64

Пропуски в датах: 14

Примеры:
дата написания письма     автор  количество слов в письме
                  NaN    Эрдман                       217
                  NaN    Эрдман                       152
           1933-11-03    Эрдман                       134
           1933-11-04 Степанова                       126
           1933